# Cómo funciona este flujo

Este pseudocódigo representa un sistema de control de acceso en una barrera. Cuando llega un vehículo, se toma una foto de la matrícula, la IA intenta reconocerla, y después se decide si se abre la barrera o si se requiere revisión manual. En la práctica, cada función correspondería a un componente real: la cámara, la inteligencia artificial, la base de datos, la garita y la barrera.

El objetivo es que el proceso quede claro: primero identificamos el vehículo, luego comprobamos si tiene permiso, y finalmente registramos la entrada o salida.


Pseudocódigo de Matriculator en Python. Describe el funcionamiento previsto después de integrar el modelo de IA. La cámara, la IA, la base de datos, la garita y la barrera se representan mediante funciones que se desarrollarían más adelante.

In [ ]:
def obtener_matricula():
    # La cámara toma una foto del vehículo cuando llega a la barrera.
    foto = hacer_foto_con_camara()

    # Si la cámara no detecta nada, no hay matrícula que analizar.
    if foto is None:
        return foto, None, "La cámara no ha enviado ninguna foto"

    # La IA intenta leer la matrícula y también dice qué tan segura es la lectura.
    matricula, confianza = leer_matricula_con_ia(foto)

    # Si la lectura falla o la confianza es baja, se requiere revisión humana.
    if matricula is None or confianza < 0.80:
        return foto, None, "No se distingue bien la matrícula"

    # Si todo sale bien, devolvemos la foto y la matrícula reconocida.
    return foto, matricula, None


In [ ]:
def decidir_acceso(matricula, problema):
    # Si ya hay un problema previo, no se sigue con la lógica del acceso.
    # Se deriva la decisión a una revisión humana en la garita.
    if problema is not None:
        return pedir_revision_en_garita(problema)

    # Buscamos si el vehículo está registrado en la base de datos.
    vehiculo = buscar_vehiculo(matricula)

    # Si no existe ese vehículo, también hay que revisarlo manualmente.
    if vehiculo is None:
        return pedir_revision_en_garita("No conocemos este vehículo")

    # Si falta información importante del vehículo, se pide revisión.
    if vehiculo.datos_incompletos:
        return pedir_revision_en_garita("Hay datos que revisar")

    # Si es docente o tiene permiso explícito, se le deja pasar.
    if vehiculo.tipo == "docente" or vehiculo.tiene_permiso:
        return "Autorizado automáticamente"

    # Si no tiene permiso ni es docente, se deniega el acceso.
    return "Denegado: no tiene permiso"


In [ ]:
def procesar_acceso():
    # El proceso empieza cuando un coche llega a la barrera y debemos decidir si abre o no.
    foto, matricula, problema = obtener_matricula()

    # Comprobamos si es un vehículo conocido y autorizado.
    resultado = decidir_acceso(matricula, problema)
    fecha_entrada = None

    # Si el resultado indica acceso válido, se abre la barrera y se registra la hora de entrada.
    if resultado in ["Autorizado automáticamente", "Autorizado manualmente"]:
        abrir_barrera()
        fecha_entrada = obtener_fecha_y_hora()
    else:
        # Si no tiene permiso o necesita revisión, la barrera sigue cerrada.
        mantener_barrera_cerrada()

    # Se guarda el intento para dejar un registro del acceso.
    guardar_intento(foto, matricula, resultado, fecha_entrada)
    mostrar_resultado(resultado)
    return resultado


In [ ]:
def registrar_salida(matricula):
    # Buscamos si ese vehículo ya entró y aún no tiene salida registrada.
    entrada = buscar_entrada_sin_salida(matricula)

    # Si no hay una entrada pendiente, no se puede registrar la salida.
    if entrada is None:
        return "No encontramos una entrada pendiente de salida"

    # Se marca la hora de salida y se guarda el cambio.
    entrada.fecha_salida = obtener_fecha_y_hora()
    guardar_cambios(entrada)
    return "Salida registrada"
